![alt text](./Cerny_logo_1.jpg)

# Analysis of Cerny ventilation recordings

## Analysis of HFOV cases `AT000001-AT001251`

This Notebook produces Excel statistics about ventilator parameters in cases ventilated with HFOV. It also exports barplots showing statistics on ventilation modes used. It does not produse graphs on individual recordings . 

### 1. Import the required libraries and set options

In [1]:
import IPython
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.dates as dates

import os
import sys
import pickle

from pandas import Series, DataFrame
from datetime import datetime, timedelta

%matplotlib inline
matplotlib.style.use('classic')
matplotlib.rcParams['figure.facecolor'] = 'w'

pd.set_option('display.max_rows', 100)
pd.set_option('display.max_columns', 100)
pd.set_option('mode.chained_assignment', None) 

In [2]:
print("Python version: {}".format(sys.version))
print("pandas version: {}".format(pd.__version__))
print("matplotlib version: {}".format(matplotlib.__version__))
print("NumPy version: {}".format(np.__version__))
print("IPython version: {}".format(IPython.__version__))

Python version: 3.12.11 | packaged by Anaconda, Inc. | (main, Jun  5 2025, 08:03:38) [Clang 14.0.6 ]
pandas version: 2.2.2
matplotlib version: 3.9.2
NumPy version: 1.26.4
IPython version: 9.11.0


### 2. List and set the working directory and the directory to write out data

In [ ]:
# Topic of the Notebook which will also be the name of the subfolder containing results
TOPIC = 'HFOV'

# Path to project folder containing clinical data (current weights only) and for export of results
PATH = os.path.join(os.sep, 'Users', 'guszti', 'Library', 'Mobile Documents', 'com~apple~CloudDocs', 
                            'Documents', 'Research', 'Ventilation')

# Name of the external hard drive
DRIVE = 'GUSZTI'

# Processed data loaded from external drive
DIR_READ = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new')

# Results will be written in this folder
DIR_WRITE =  os.path.join(os.sep, PATH, 'ventilation_fabian_new', 'Analyses', TOPIC)
os.makedirs(DIR_WRITE, exist_ok=True)

# Images and raw data will be written on an external hard drive
DATA_DUMP = os.path.join(os.sep, 'Volumes', DRIVE, 'data_dump', 'fabian_new', TOPIC)
os.makedirs(DATA_DUMP, exist_ok=True)

In [ ]:
DIR_READ, DIR_WRITE, DATA_DUMP

### 3. Import ventilator data from pickle archives

In [ ]:
# Import ventilator parameters, settings and alarms

with open(os.path.join(DIR_READ, 'data_pars_measurements_trimmed_new.pickle'), 'rb') as handle:
    data_pars_measurements_trimmed = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_settings_trimmed_new.pickle'), 'rb') as handle:
    data_pars_settings_trimmed = pickle.load(handle)
with open(os.path.join(DIR_READ, 'data_pars_alarms_trimmed_new.pickle'), 'rb') as handle:
    data_pars_alarms_trimmed = pickle.load(handle)

In [ ]:
len(data_pars_measurements_trimmed), len(data_pars_settings_trimmed), len(data_pars_alarms_trimmed)

In [ ]:
# Import DataFrame with ventilation modes

with open(os.path.join(DIR_READ, 'vent_modes_trimmed_new.pickle'), 'rb') as handle:
    vent_modes_trimmed_new = pickle.load(handle)

vent_modes_trimmed_new

### 4. Shift the time stamp of ventilator recordings recorded with incorrect time stamps

Az eltérés valóban az AT000110-estől kezdődik és az AT000216-ig tart. Sajnos volt közben egy téli/nyári váltás is. 
Március 28 után a plusz 9 órával kell korrigálni, előtte 10 órával. 

In [ ]:
data_pars_measurements_trimmed['AT000110'].head()

In [ ]:
for case in data_pars_measurements_trimmed:
    # print(int(case[2:].lstrip('0')))
    if 110 <= int(case[2:].lstrip('0')) < 195:
        data_pars_measurements_trimmed[case].index = \
            data_pars_measurements_trimmed[case].index.shift(10, freq='H')
        data_pars_settings_trimmed[case].index = \
            data_pars_settings_trimmed[case].index.shift(10, freq='H')
        data_pars_alarms_trimmed[case].index = \
            data_pars_alarms_trimmed[case].index.shift(10, freq='H')
        
    elif 195 <= int(case[2:].lstrip('0')) <= 216:
        data_pars_measurements_trimmed[case].index = \
            data_pars_measurements_trimmed[case].index.shift(9, freq='H')
        data_pars_settings_trimmed[case].index = \
            data_pars_settings_trimmed[case].index.shift(9, freq='H')
        data_pars_alarms_trimmed[case].index = \
            data_pars_alarms_trimmed[case].index.shift(9, freq='H')

In [ ]:
data_pars_measurements_trimmed['AT000110'].head()

### 5. Identify which recordings had HFO mode

In [ ]:
vent_modes_hfov = vent_modes_trimmed_new[vent_modes_trimmed_new['HFO'] > 0]
len(vent_modes_hfov)

In [ ]:
vent_modes_hfov.sort_values(['HFO'])

In [ ]:
# Removes those recordings which have <1 minute HFOV and otherwise are noninvasive recordings 
# (these patients did not actually have HFOV
vent_modes_hfov = vent_modes_hfov[vent_modes_hfov['HFO'] > 60]
len(vent_modes_hfov)

In [ ]:
# How many had conventional ventilation as well
conv_modes = ['IPPV', 'PSV', 'SIMV', 'SIMVPSV', 'SIPPV']
vent_modes_hfov['conventional'] = vent_modes_hfov[conv_modes].sum(axis=1)

In [ ]:
vent_modes_hfov.head()

### 6. Limit data to cases who were ventilated using HFOV for at least part of the transfer

In [ ]:
# Limit ventilator recordings to hfov
data_pars_measurements_hfov = {key:value for key, value in data_pars_measurements_trimmed.items()
                               if key in vent_modes_hfov.index}
data_pars_settings_hfov = {key:value for key, value in data_pars_settings_trimmed.items()
                               if key in vent_modes_hfov.index}
data_pars_alarms_hfov = {key:value for key, value in data_pars_alarms_trimmed.items()
                               if key in vent_modes_hfov.index}

In [ ]:
len(data_pars_measurements_hfov), len(data_pars_settings_hfov), len(data_pars_alarms_hfov)

### 7. Clean up ventilator data

#### A. Clean up ventilator parameters

In [ ]:
for case in data_pars_measurements_hfov:
    # Remove completely empty columns of parameters which are not relevant
    data_pars_measurements_hfov[case].dropna(how = 'all', axis = 1, inplace = True)
    
    # Make sure the type of all remaining data is float
    data_pars_measurements_hfov[case] = data_pars_measurements_hfov[case].astype('float')

In [ ]:
data_pars_measurements_hfov['AT000033'];

#### B. Clean up ventilator settings

In [ ]:
data_pars_settings_hfov['AT000033'].head()

In [ ]:
for case in data_pars_settings_hfov:
    # Remove completely empty columns of parameters which are not relevant
    data_pars_settings_hfov[case].dropna(how = 'all', axis = 1, inplace = True)

In [ ]:
# Make categorical variables "category" type

categorical = {'Patient_range', 'Ventilator_mode', 'Powerstate', 'Measuring_unit_pressure_set', 
               'Measuring_unit_CO2_set', 'Flow_sensor_state', 'Oxy_sensor_state',
               'Ventilation_stopped', 'VG_state',  'Trigger_mode', 'I_E_HFOV', 
               'Pressure_rise_control', 'HFO_flow', 'Trigger_mode_2', 'FOT_running', 'Leak_comp'}

for case in data_pars_settings_hfov:
    to_change_to_categorical = categorical & set(data_pars_settings_hfov[case].columns)
    for par in to_change_to_categorical:
        data_pars_settings_hfov[case][par] = data_pars_settings_hfov[case][par].astype('category')

#### C. Cleanup ventilator alarms

In [ ]:
for case in data_pars_alarms_hfov:
    # Remove empty columns of parameters which are not noninvasive ventilation
    data_pars_alarms_hfov[case].dropna(how = 'all', axis = 1, inplace = True)
    # Make sure the type of all remaining data is float
    data_pars_alarms_hfov[case] = data_pars_alarms_hfov[case].astype('float')

### 8. Generate DataFrames with data limited to the hfov periods

In [ ]:
data_pars_settings_hfov['AT000456']['Ventilator_mode'].value_counts()

In [ ]:
### Generate DataFrames with the HFO parts only 
data_pars_settings_hfov_only = {}
data_pars_measurements_hfov_only = {}
data_pars_alarms_hfov_only = {}

for recording in data_pars_settings_hfov:
    data_pars_settings_hfov_only[recording] = \
        data_pars_settings_hfov[recording][data_pars_settings_hfov[recording]['Ventilator_mode'] == 'HFO']
    
    to_reindex_with = data_pars_settings_hfov_only[recording].index
    
    data_pars_measurements_hfov_only[recording] = data_pars_measurements_hfov[recording].reindex(to_reindex_with)
    data_pars_alarms_hfov_only[recording] = data_pars_alarms_hfov[recording].reindex(to_reindex_with)

### 9. Calculate difference between set and actual parameters

In [ ]:
hfov_pars_settings = [('deltaP', 'deltaP_set'), ('HFOV_freq', 'Freq_set_HFOV'), ('FiO2', 'FiO2_set'),
                      ('MAP', 'MAP_set_HFOV')]

for par, setting in hfov_pars_settings:
    for case in data_pars_measurements_hfov_only:
        data_pars_measurements_hfov_only[case][f'{par}_diff'] = \
            data_pars_measurements_hfov_only[case][par] - data_pars_settings_hfov_only[case][setting]

for case in data_pars_measurements_hfov_only:
    
    if 'VG_set' in data_pars_settings_hfov_only[case]:
        data_pars_measurements_hfov_only[case]['VThf_diff'] = data_pars_measurements_hfov_only[case][par] - \
            data_pars_settings_hfov_only[case]['VG_set'][data_pars_settings_hfov_only[case]['VG_set'] != 'off']
    
    if  'VG_set_kg' in data_pars_settings_hfov_only[case]: 
        data_pars_measurements_hfov_only[case]['VThf_diff_kg'] = data_pars_measurements_hfov_only[case][par] - \
            data_pars_settings_hfov_only[case]['VG_set_kg'][data_pars_settings_hfov_only[case]['VG_set_kg'] != 'off']

### 10. Resample the measurements and the settings data to calculate 1-minute means

In [ ]:
data_pars_measurements_hfov_1min_mean = {}
data_pars_settings_hfov_1min_mean = {}

data_pars_measurements_hfov_only_1min_mean = {}
data_pars_settings_hfov_only_1min_mean = {}

for case in data_pars_measurements_hfov:
    data_pars_measurements_hfov_1min_mean[case] = \
        data_pars_measurements_hfov[case].resample('1min').mean()
    data_pars_settings_hfov_1min_mean[case] = \
        data_pars_settings_hfov[case].resample('1min').mean()
    
    data_pars_measurements_hfov_only_1min_mean[case] = \
        data_pars_measurements_hfov_only[case].resample('1min').mean()
    data_pars_settings_hfov_only_1min_mean[case] = \
        data_pars_settings_hfov_only[case].resample('1min').mean()

### 11. Export processed data as pickle arcives

In [ ]:
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_measurements_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_measurements_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

with open(os.path.join(DATA_DUMP, 'data_pars_settings_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_settings_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)
    
with open(os.path.join(DATA_DUMP, 'data_pars_alarms_hfov_only.pickle'), 'wb') as handle:
    pickle.dump(data_pars_alarms_hfov_only, handle, protocol=pickle.HIGHEST_PROTOCOL)

In [ ]:
with open(os.path.join(DATA_DUMP, 'vent_modes_hfov.pickle'), 'wb') as handle:
    pickle.dump(vent_modes_hfov, handle, protocol=pickle.HIGHEST_PROTOCOL)

### 12. Generate and export graphs for ventilator recordings containing HFOV

Export both the whole recordings and only HFOV parts

In [ ]:
os.makedirs(os.path.join(DIR_WRITE, 'graphs'), exist_ok = True)

#### A. FiO2

##### Whole recording

In [ ]:
%%time

par = 'FiO2'
dim = '%'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(0, 110)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['FiO$_2$'], loc=4)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

##### HFOV part only

In [ ]:
%%time

par = 'FiO2'
dim = '%'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(0, 110)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['FiO$_2$'], loc=4)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);

    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### B. MAP

##### Whole recording

In [ ]:
%%time

par = 'MAP'
dim = 'cmH$_2$O'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(0, 30)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['MAP'], loc=4)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

##### HFOV part only

In [ ]:
%%time

pars = ['MAP', 'MAP_set_HFOV',]
name = 'MAPhf_MAPhf_set'
dim = 'cmH$_2$O'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only:
    
    #print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax0 = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][pars[0]].plot(color = 'red', x_compat = True)
    data_pars_settings_hfov_only[case][pars[1]].plot(color = 'black', linewidth = 2, 
                                                     linestyle='dashed', x_compat = True)
    
    ax0.set_ylim(0, data_pars_settings_hfov_only[case][pars[1]].max() + 10)
    ax0.set_xlabel('Time', size = 14, color = 'black')
    ax0.set_ylabel(dim, size = 14, color = 'black')
    ax0.set_title(case,  size = 14, color = 'black')
    ax0.legend(['MAPhf', 'MAPhf_set'], loc=4, fontsize = 12)
    ax0.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax0.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax0.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax0.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### C. deltaP and deltaP_set

##### HFOV part only

In [ ]:
%%time

pars = ['deltaP', 'deltaP_set',]
name = 'deltaP_deltaP_set'
dim = 'cmH$_2$O'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only:
    
    #print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax0 = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][pars[0]].plot(color = 'red', x_compat = True)
    data_pars_settings_hfov_only[case][pars[1]].plot(color = 'black', linewidth = 2, 
                                                     linestyle='dashed', x_compat = True)
    
    ax0.set_ylim(0, data_pars_settings_hfov_only[case][pars[1]].max() + 20)
    ax0.set_xlabel('Time', size = 14, color = 'black')
    ax0.set_ylabel(dim, size = 14, color = 'black')
    ax0.set_title(case,  size = 14, color = 'black')
    ax0.legend(['deltaP', 'deltaP_set'], loc=4, fontsize = 12)
    ax0.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax0.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax0.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax0.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### D. Frequency

##### HFOV part only

In [ ]:
%%time

pars = ['HFOV_freq', 'Freq_set_HFOV',]
name = 'HFOV_freq_HFOV_freq_set'
dim = 'Hz'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only:
    
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax0 = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][pars[0]].plot(color = 'blue', x_compat = True)
    data_pars_settings_hfov_only[case][pars[1]].plot(color = 'black', linewidth = 2, 
                                                linestyle='dashed', x_compat = True)
    
    ax0.set_ylim(0, 18)
    ax0.set_xlabel('Time', size = 14, color = 'black')
    ax0.set_ylabel(dim, size = 14, color = 'black')
    ax0.set_title(case,  size = 14, color = 'black')
    ax0.legend(['frequency', 'frequency_set'], loc=4, fontsize = 12)
    ax0.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax0.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax0.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax0.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### E. VThf

##### HFOV part only

In [ ]:
%%time

par = 'VThf_emand_kg'
dim = 'mL/kg'; filetype = 'jpg'; dpi = 200
name = 'VThf'

for case in data_pars_measurements_hfov_only:
    
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(0, 6)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['VThf'], loc=4, fontsize = 12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{name}_hfov_only_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### F. VTHf and VThf_set

##### HFOV+VG part only

In [ ]:
%%time

pars = ['VThf_emand_kg', 'VG_set_kg',]
name = 'VThf_VThf_set'
dim = 'mL/kg'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only: 
    if pars[1] not in data_pars_settings_hfov_only[case].columns: # VG not on
        continue
    
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    ax0 = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][pars[0]].plot(color = 'blue', x_compat = True)
    data_pars_settings_hfov_only[case][pars[1]].plot(color = 'black', linewidth = 2,
                                                     linestyle='dashed', x_compat = True)
    
    ax0.set_ylim(0, 6)
    ax0.set_xlabel('Time', size = 14, color = 'black')
    ax0.set_ylabel(dim, size = 14, color = 'black')
    ax0.set_title(case,  size = 14, color = 'black')
    ax0.legend(['VThf', 'VThf_set'], loc=1, fontsize = 12)
    ax0.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax0.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax0.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax0.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### G. DCO2

##### HFOV part only

In [ ]:
%%time

par = 'DCO2_kg2'
dim = r'mL$^2$/s/kg$^2$'; filetype = 'jpg'; dpi = 200
name = 'DCO2'

for case in data_pars_measurements_hfov_only:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(0, 100)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['DCO$_2$'], loc=1, fontsize=12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### H. Leak

##### Whole recording

In [ ]:
%%time

par = 'Leak'
dim = '%'; filetype = 'pdf'; dpi = 200

for case in data_pars_measurements_hfov:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(-5, 100)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend([par], loc=1, fontsize=12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

##### HFOV part only

In [ ]:
%%time

par = 'Leak'
dim = '%'; filetype = 'jpg'; dpi = 200

for case in data_pars_measurements_hfov_only:
    # print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    data_pars_measurements_hfov_only[case][par].plot(ax = ax, x_compat = True)
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel(dim, size = 14, color = 'black')
    ax.set_ylim(-5, 100)
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend([par], loc=1, fontsize=12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_only_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_only_{par}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()

#### I. Ventilator modes

In [ ]:
mapping_vent_mode = {1: 'IPPV', 2: 'SIPPV', 3: 'SIMV', 4: 'SIMVPSV', 5: 'PSV', 
                     6: 'CPAP', 7: 'NCPAP', 8: 'DUOPAP', 9: 'HFO', 10: 'O2therapy'}
mapping_vent_mode_rev = {value:key for key, value in mapping_vent_mode.items()}
mapping_vent_mode_rev

In [ ]:
mapping_VG_state = {12: 'on', 13: 'off'}
mapping_VG_state_rev = {value:key for key, value in mapping_VG_state.items()}
mapping_VG_state_rev

In [ ]:
%%time

par = 'Ventilator_mode'
dim = 'cmH$_2$O'; filetype = 'jpg'; dpi = 200

for case in data_pars_settings_hfov:
    #print('Saving %s' % case)
    fig = plt.figure()
    fig.set_size_inches(8, 4)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    ax = fig.add_subplot(1, 1, 1)
    
    data = data_pars_settings_hfov[case][par].replace(mapping_vent_mode_rev).astype('float').dropna()
    data.plot(ax = ax, color='black', linewidth=2, x_compat = True)
    
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel('', size = 14, color = 'black')
    ax.set_ylim(0,14)
    ax.set_yticks(sorted(mapping_vent_mode_rev.values()))
    ax.set_yticklabels(mapping_vent_mode_rev.keys())
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['Ventilation mode'], fontsize =12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
            
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_{par}.{filetype}'), 
    dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_{par}.{filetype}'), 
    dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()
    

In [ ]:
%%time

pars = ['Ventilator_mode', 'VG_state']
dim = 'cmH$_2$O'; filetype = 'jpg'; dpi = 200
name = 'Ventilator_mode_VG_state'

for case in data_pars_settings_hfov:
    
    #print('Saving %s' % case)
    
    fig = plt.figure()
    fig.set_size_inches(8, 6)
    fig.subplots_adjust(left=None, bottom=None, right=None, top=None, wspace=None, hspace=0.1)
    
    ax = fig.add_subplot(1, 1, 1)
    
    data = data_pars_settings_hfov[case][pars[0]].replace(mapping_vent_mode_rev).astype('float').dropna()
    data.plot(ax = ax, color='black', linewidth=2, x_compat = True)
    
    if 'VG_state' in data_pars_settings_hfov[case].columns:
        data_2 = data_pars_settings_hfov[case][pars[1]].replace(mapping_VG_state_rev).astype('float').dropna()
        data_2.plot(ax = ax, color='black', linewidth=2, linestyle='dashed', x_compat = True)
    
    else:
        plt.axhline(13, linestyle='dashed', linewidth=2, color='black')
        
    ax.set_xlabel('Time', size = 14, color = 'black')
    ax.set_ylabel('', size = 14, color = 'black')
    ax.set_ylim(0,17)
    ax.set_yticks(sorted(mapping_vent_mode_rev.values()) + [11] + sorted(mapping_VG_state_rev.values()))
    ax.set_yticklabels(list(mapping_vent_mode_rev.keys()) + [' '] + list(mapping_VG_state_rev.keys()))
    ax.set_title(case,  size = 14, color = 'black')
    ax.legend(['Ventilation mode', 'VG'], fontsize=12)
    ax.grid('on', linestyle='-', linewidth=0.5, color = 'gray')
    
    majorFmt = dates.DateFormatter('%H:%M')  
    ax.xaxis.set_major_formatter(majorFmt)
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, fontsize = 12, horizontalalignment = 'center')
    ax.tick_params(which = 'both', labelsize=12)
         
    fig.savefig(os.path.join(DIR_READ, 'fabian_cases_new', case, f'{case}_hfov_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    fig.savefig(os.path.join(DIR_WRITE, 'graphs', f'{case}_hfov_{name}.{filetype}'), 
        dpi = dpi, format = filetype, bbox_inches='tight', pad_inches=0.1,);
    
    if case != 'AT000591':
        plt.close()
    